In [5]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [7]:
import pandas as pd

In [9]:
pip install peft==0.9.0

Note: you may need to restart the kernel to use updated packages.


In [10]:
!pip install huggingface_hub[hf_xet]

In [12]:
!pip install git+https://github.com/tabularis-ai/be_great.git

  Cloning https://github.com/tabularis-ai/be_great.git to c:\users\haava\appdata\local\temp\pip-req-build-j4jhf2yc
  Resolved https://github.com/tabularis-ai/be_great.git to commit 468329e6ef34a5908ad639cf4df1ff654a27799e
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/tabularis-ai/be_great.git 'C:\Users\haava\AppData\Local\Temp\pip-req-build-j4jhf2yc'


In [14]:
import be_great.metrics

In [15]:
df = pd.read_csv(r"C:\Users\haava\Documents\student\student-mat.csv", sep=";")

In [16]:
drop_cols = [
    "G1",
    "G2",
    "guardian",
    "reason",
    "nursery",
    "romantic",
    "schoolsup"
]

df = df.drop(columns=drop_cols)


for c in df.select_dtypes(include="object"):
    df[c] = (
        df[c]
        .astype(str)
        .str.replace(",", " ", regex=False)
        .str.replace("\n", " ", regex=False)
        .str.strip()
    )

df = df.fillna("")

print(df.shape)  

(395, 26)


In [19]:
categorical = df.select_dtypes(include="object").columns
numeric = df.select_dtypes(exclude="object").columns

df = df[list(categorical) + list(numeric)]

In [20]:
from sklearn.model_selection import train_test_split

D_real_train, D_real_holdout = train_test_split(
    df,
    train_size=300,
    test_size=95,
    random_state=42,
    shuffle=True
)

In [21]:
df.head().transpose()

,0,1,2,3,4
school,GP,GP,GP,GP,GP
sex,F,F,F,F,F
address,U,U,U,U,U
famsize,GT3,GT3,LE3,GT3,GT3
Pstatus,A,T,T,T,T
Mjob,at_home,at_home,at_home,health,other
Fjob,teacher,other,other,services,other
famsup,no,yes,no,yes,yes
paid,no,no,yes,yes,yes
activities,no,no,no,yes,no


In [22]:
D_real_train.shape

(300, 26)

In [33]:
import peft
print(peft.__version__)

0.9.0


In [35]:
from be_great import GReaT

model = GReaT(
    llm="facebook/opt-350m",
    batch_size=8,
    epochs=40,
    efficient_finetuning="lora",
    lora_config={
        "r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
        "target_modules": ["q_proj", "v_proj"],
    },
    fp16=True,
)

model.fit(D_real_train)

trainable params: 786,432 || all params: 331,982,848 || trainable%: 0.2368893467652883


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
500,1.757700
1000,1.043700
1500,0.994200


In [36]:
model.tokenizer.pad_token = model.tokenizer.eos_token
model.model.config.pad_token_id = model.tokenizer.pad_token_id
synthetic = model.sample(
    n_samples=300,
    max_length=200,
    temperature=0.7,
    guided_sampling=True
)

print("Generated:", len(synthetic))
synthetic.to_csv("synthetic_students_opt250.csv", index=False)

100%|████████████████████████████████████████████████████████████████████████████████| 300/300 [56:50<00:00, 11.37s/it]

Generated: 300


In [39]:
df_syn_opt250 = pd.read_csv("synthetic_students_opt250.csv", sep=",")

In [41]:
df_syn_opt250.shape


(300, 26)

In [43]:
df_syn_opt250.head().transpose()

,0,1,2,3,4
school,GP,GP,GP,GP,GP
sex,MS,F,M,M,M
address,U,U,U,U,U
famsize,GT3,GT3,GT3,GT3,GT3
Pstatus,T,T,T,T,T
Mjob,other,at_home,services,at_home,other
Fjob,other,other,services,at_home,other
famsup,yes,yes,yes,yes,yes
paid,"yes, higher is yes",yes,yes,no,yes
activities,yes,yes,no,yes,no


In [45]:
df.head().transpose()

,0,1,2,3,4
school,GP,GP,GP,GP,GP
sex,F,F,F,F,F
address,U,U,U,U,U
famsize,GT3,GT3,LE3,GT3,GT3
Pstatus,A,T,T,T,T
Mjob,at_home,at_home,at_home,health,other
Fjob,teacher,other,other,services,other
famsup,no,yes,no,yes,yes
paid,no,no,yes,yes,yes
activities,no,no,no,yes,no


In [47]:
def cast_synthetic_to_original_schema(synth: pd.DataFrame, original: pd.DataFrame) -> pd.DataFrame:
    synth = synth.copy()

    synth = synth[[c for c in original.columns if c in synth.columns]]

    for col in synth.columns:
        orig_col = original[col]


        if pd.api.types.is_object_dtype(orig_col) or pd.api.types.is_string_dtype(orig_col):
            synth[col] = synth[col].astype(str)

        elif pd.api.types.is_integer_dtype(orig_col):
            synth[col] = pd.to_numeric(synth[col], errors="coerce")
         
            fill_value = int(orig_col.median())
            synth[col] = synth[col].fillna(fill_value).round().astype(orig_col.dtype)

        elif pd.api.types.is_float_dtype(orig_col):
            synth[col] = pd.to_numeric(synth[col], errors="coerce")

            fill_value = float(orig_col.median())
            synth[col] = synth[col].fillna(fill_value).astype(orig_col.dtype)
 
        elif pd.api.types.is_bool_dtype(orig_col):
            mapping = {
                "true": True, "false": False,
                "1": True, "0": False,
                "yes": True, "no": False
            }
            synth[col] = (
                synth[col]
                .astype(str)
                .str.strip()
                .str.lower()
                .map(mapping)
                .fillna(False)
                .astype(bool)
            )

        else:

            synth[col] = synth[col].astype(orig_col.dtype, errors="ignore")

    return synth

synthetic_casted = cast_synthetic_to_original_schema(df_syn_opt250, df)

print(synthetic_casted.dtypes)
synthetic_casted.to_csv("synthetic_students_typed_opt250.csv", index=False)

school        object
sex           object
address       object
famsize       object
Pstatus       object
Mjob          object
Fjob          object
famsup        object
paid          object
activities    object
higher        object
internet      object
age            int64
Medu           int64
Fedu           int64
traveltime     int64
studytime      int64
failures       int64
famrel         int64
freetime       int64
goout          int64
Dalc           int64
Walc           int64
health         int64
absences       int64
G3             int64
dtype: object


In [49]:
df_syn_opt250_typed = pd.read_csv("synthetic_students_typed_opt250.csv", sep=",")

In [51]:
df_syn_opt250_typed.shape

(300, 26)

In [53]:
df_syn_opt250_typed.head()

,school,sex,address,famsize,Pstatus,Mjob,Fjob,famsup,paid,activities,...,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G3
0,GP,MS,U,GT3,T,other,other,yes,"yes, higher is yes",yes,...,2,0,3,3,4,2,1,3,1,10
1,GP,F,U,GT3,T,at_home,other,yes,yes,yes,...,2,0,3,1,4,1,1,5,0,10
2,GP,M,U,GT3,T,services,services,yes,yes,no,...,3,0,4,3,3,2,4,4,8,0
3,GP,M,U,GT3,T,at_home,at_home,yes,no,yes,...,2,1,4,3,4,2,1,4,3,8
4,GP,M,U,GT3,T,other,other,yes,yes,no,...,1,0,3,4,3,1,1,5,0,11
